In [6]:
import argparse
import os
import re
import requests

def parse_rfam_tree_for_mapping(rfam_tree_text):
    """
    Parse Rfam tree to create mapping: accession → species_name

    Rfam label: _URS000063031F_10116/1-75_Rattus_norvegicus_{Norw..[10116].1
    We want:    URS000063031F_10116/1-75 → Rattus_norvegicus
    """
    acc_to_species = {}

    # Pattern: accession/coords followed by species name (Genus_species)
    pattern = r'_?([A-Z0-9]+_\d+/\d+-\d+)_([A-Z][a-z]+_[a-z]+)'

    for match in re.finditer(pattern, rfam_tree_text):
        accession = match.group(1)
        species = match.group(2)
        acc_to_species[accession] = species

    print(f"  Found {len(acc_to_species)} accession → species mappings")

    return acc_to_species, accession, species

rfam_RF00740 = '((_URS000075E9DB_9796/1-75_Equus_caballus_{horse}[9796].1:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_..[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta_{Rhesus_..[9544].1:0.00053,((_URS000075E3EC_9913/1-75_Bos_taurus_{cattle}[9913].1:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries_{sheep}[9940].1:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens_{human}[9606].1:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus_{Bornean..[9600].1:0.0,_URS0000A95B56_9823/2-76_Sus_scrofa_{pig}[9823].1:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes_{chimpa..[9598].1:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus_{big_b..[29078].1:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus_{house_mou..[10090].1:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus_{Norw..[10116].1:0.00055)0.940:0.02717)0.140:0.00055);%'

acc_to_spec, acc, spec = parse_rfam_tree_for_mapping(rfam_RF00740)
print(acc_to_spec)
print('===================')
print(acc)
print('===================')
print(spec)

  Found 12 accession → species mappings
{'URS000075E9DB_9796/1-75': 'Equus_caballus', 'URS000027000B_9615/1-75': 'Canis_lupus', 'URS000075D5FE_9544/1-75': 'Macaca_mulatta', 'URS000075E3EC_9913/1-75': 'Bos_taurus', 'URS000075BA4A_9940/13-87': 'Ovis_aries', 'URS000075B235_9606/1-75': 'Homo_sapiens', 'URS000075B235_9600/1-75': 'Pongo_pygmaeus', 'URS0000A95B56_9823/2-76': 'Sus_scrofa', 'URS0000759D1D_9598/1-74': 'Pan_troglodytes', 'URS0000D514AB_29078/1-77': 'Eptesicus_fuscus', 'URS0000662F5D_10090/1-75': 'Mus_musculus', 'URS000063031F_10116/1-75': 'Rattus_norvegicus'}
URS000063031F_10116/1-75
Rattus_norvegicus


In [3]:
try:
    import dendropy

    HAS_DENDROPY = True
except ImportError:
    HAS_DENDROPY = False
    print("WARNING: dendropy not installed. Install with: pip install dendropy")

def convert_tree_labels(tree_newick, acc_to_species):
    """
    Convert tree labels from accessions to species names.

    Input label:  URS000063031F_10116/1-75
    Output label: Rattus_norvegicus
    """
    if not HAS_DENDROPY:
        print("ERROR: dendropy required")
        return None

    try:
        tree = dendropy.Tree.get(data=tree_newick, schema="newick")

        converted = 0
        failed = []

        for leaf in tree.leaf_node_iter():
            if leaf.taxon is None:
                continue

            label = leaf.taxon.label

            # Try direct lookup
            if label in acc_to_species:
                leaf.taxon.label = acc_to_species[label]
                converted += 1
                continue

            # Try to extract accession pattern from label
            match = re.search(r'([A-Z0-9]+_\d+/\d+-\d+)', label)
            if match:
                acc = match.group(1)
                if acc in acc_to_species:
                    leaf.taxon.label = acc_to_species[acc]
                    converted += 1
                    continue

            failed.append(label)

        if failed:
            print(f"    Warning: {len(failed)} labels not found in mapping")

        return tree.as_string(schema="newick")

    except Exception as e:
        print(f"  Error: {e}")
        return None


print(convert_tree_labels(rfam_RF00740, acc_to_spec))


  Error: Error parsing data source on line 1 at column 43: Expecting ':', ')', ',' or ';' after reading label but found '{'
None


In [7]:
def clean_rfam_tree_for_parsing(rfam_tree_text):
    """
    Clean Rfam tree so dendropy can parse it.

    Problem: Labels like _URS000075E9DB_9796/1-75_Equus_caballus_{horse}[9796].1
    contain { } which dendropy can't handle.

    Solution: Remove everything from { to the next ] (i.e., remove {horse}[9796].1)
    """
    # Remove {common_name}[taxid].1 patterns
    cleaned = re.sub(r'\{[^}]*\}\[\d+\]\.\d+', '', rfam_tree_text)
    # Also handle if there's just {something} without the rest
    cleaned = re.sub(r'\{[^}]*\}', '', cleaned)
    # Remove trailing dots if any
    cleaned = re.sub(r'\.\.+', '', cleaned)
    return cleaned

clean_rfam_tree_for_parsing(rfam_RF00740)

'((_URS000075E9DB_9796/1-75_Equus_caballus_:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta_:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries_:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens_:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus_:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes_{chimpa[9598].1:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus_{big_b[29078].1:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus_{house_mou[10090].1:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus_{Norw[10116].1:0.00055)0.940:0.02717)0.140:0.00055);%'

In [16]:
clean1 = re.sub(r'\_{[^.}]*\}\[\d+\]\.\d+', '', rfam_RF00740)
re.sub(r'_{[^:]+', '', clean1)

'((_URS000075E9DB_9796/1-75_Equus_caballus:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_..[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta:0.00053,((_URS000075E3EC_9913/1-75_Bos_taurus:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus:0.0,_URS0000A95B56_9823/2-76_Sus_scrofa:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus:0.00055)0.940:0.02717)0.140:0.00055);%'

In [10]:
rfam_RF00740

'((_URS000075E9DB_9796/1-75_Equus_caballus_{horse}[9796].1:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_..[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta_{Rhesus_..[9544].1:0.00053,((_URS000075E3EC_9913/1-75_Bos_taurus_{cattle}[9913].1:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries_{sheep}[9940].1:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens_{human}[9606].1:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus_{Bornean..[9600].1:0.0,_URS0000A95B56_9823/2-76_Sus_scrofa_{pig}[9823].1:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes_{chimpa..[9598].1:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus_{big_b..[29078].1:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus_{house_mou..[10090].1:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus_{Norw..[10116].1:0.00055)0.940:0.02717)0.140:0.00055);%'

In [15]:
clean1

'((_URS000075E9DB_9796/1-75_Equus_caballus:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_..[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta_{Rhesus_..[9544].1:0.00053,((_URS000075E3EC_9913/1-75_Bos_taurus:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus_{Bornean..[9600].1:0.0,_URS0000A95B56_9823/2-76_Sus_scrofa:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes_{chimpa..[9598].1:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus_{big_b..[29078].1:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus_{house_mou..[10090].1:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus_{Norw..[10116].1:0.00055)0.940:0.02717)0.140:0.00055);%'

In [23]:
rfam_RF00740_clean = re.sub(r'_{[^:]+', '', rfam_RF00740)

rfam_RF00740_clean


'((_URS000075E9DB_9796/1-75_Equus_caballus:0.0,_URS000027000B_9615/1-75_Canis_lupus_familiaris_..[9615].1:0.0):0.00055,(_URS000075D5FE_9544/1-75_Macaca_mulatta:0.00053,((_URS000075E3EC_9913/1-75_Bos_taurus:0.00055,_URS000075BA4A_9940/13-87_Ovis_aries:0.01343)0.900:0.02718,((_URS000075B235_9606/1-75_Homo_sapiens:0.0,_URS000075B235_9600/1-75_Pongo_pygmaeus:0.0,_URS0000A95B56_9823/2-76_Sus_scrofa:0.0):0.00055,_URS0000759D1D_9598/1-74_Pan_troglodytes:0.00055)0.630:0.00054)0.830:0.01343)0.860:0.01344,(_URS0000D514AB_29078/1-77_Eptesicus_fuscus:0.01405,(_URS0000662F5D_10090/1-75_Mus_musculus:0.01343,_URS000063031F_10116/1-75_Rattus_norvegicus:0.00055)0.940:0.02717)0.140:0.00055);%'

In [24]:
tree = dendropy.Tree.get(data=rfam_RF00740_clean, schema="newick")

for leaf in tree.leaf_node_iter():
    if leaf.taxon is None:
        continue
    label = leaf.taxon.label

    for label in acc_to_spec:
        leaf.taxon.label = acc_to_spec[label]


URS000075E9DB_9796/1-75
URS000027000B_9615/1-75
URS000075D5FE_9544/1-75
URS000075E3EC_9913/1-75
URS000075BA4A_9940/13-87
URS000075B235_9606/1-75
URS000075B235_9600/1-75
URS0000A95B56_9823/2-76
URS0000759D1D_9598/1-74
URS0000D514AB_29078/1-77
URS0000662F5D_10090/1-75
URS000063031F_10116/1-75
URS000075E9DB_9796/1-75
URS000027000B_9615/1-75
URS000075D5FE_9544/1-75
URS000075E3EC_9913/1-75
URS000075BA4A_9940/13-87
URS000075B235_9606/1-75
URS000075B235_9600/1-75
URS0000A95B56_9823/2-76
URS0000759D1D_9598/1-74
URS0000D514AB_29078/1-77
URS0000662F5D_10090/1-75
URS000063031F_10116/1-75
URS000075E9DB_9796/1-75
URS000027000B_9615/1-75
URS000075D5FE_9544/1-75
URS000075E3EC_9913/1-75
URS000075BA4A_9940/13-87
URS000075B235_9606/1-75
URS000075B235_9600/1-75
URS0000A95B56_9823/2-76
URS0000759D1D_9598/1-74
URS0000D514AB_29078/1-77
URS0000662F5D_10090/1-75
URS000063031F_10116/1-75
URS000075E9DB_9796/1-75
URS000027000B_9615/1-75
URS000075D5FE_9544/1-75
URS000075E3EC_9913/1-75
URS000075BA4A_9940/13-87
URS

In [25]:
acc_to_spec

{'URS000075E9DB_9796/1-75': 'Equus_caballus',
 'URS000027000B_9615/1-75': 'Canis_lupus',
 'URS000075D5FE_9544/1-75': 'Macaca_mulatta',
 'URS000075E3EC_9913/1-75': 'Bos_taurus',
 'URS000075BA4A_9940/13-87': 'Ovis_aries',
 'URS000075B235_9606/1-75': 'Homo_sapiens',
 'URS000075B235_9600/1-75': 'Pongo_pygmaeus',
 'URS0000A95B56_9823/2-76': 'Sus_scrofa',
 'URS0000759D1D_9598/1-74': 'Pan_troglodytes',
 'URS0000D514AB_29078/1-77': 'Eptesicus_fuscus',
 'URS0000662F5D_10090/1-75': 'Mus_musculus',
 'URS000063031F_10116/1-75': 'Rattus_norvegicus'}